# MiniT2I Diffusers Demo

This notebook samples from the Hugging Face Diffusers checkpoints for **MiniT2I-B/16** and **MiniT2I-L/16**. The checkpoint repository is private; run the authentication cell with a Hugging Face token that has access to the `MiniT2I` organization.

[GitHub](https://github.com/Hope7Happiness/minit2i-torch) | [Hugging Face](https://huggingface.co/MiniT2I/MiniT2I)


# 1. Setup

Use a GPU runtime: **Runtime > Change runtime type > Hardware accelerator > GPU**. A T4 runtime is enough for MiniT2I-B/16; MiniT2I-L/16 is slower and uses more memory.


In [ ]:
%%capture
!pip install -U -q diffusers transformers accelerate safetensors huggingface_hub


In [ ]:
#@title Runtime setup
import os

os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("TRANSFORMERS_NO_FLAX", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
from IPython.display import display

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"device={device}, dtype={dtype}")
if device == "cpu":
    print("GPU not found. Generation will be very slow; switch the Colab runtime to GPU.")


# 2. Authenticate

Recommended in Colab: open the key icon in the left sidebar, add a secret named `HF_TOKEN`, and enable notebook access. If no secret is found, this cell opens the standard Hugging Face notebook login widget.


In [ ]:
#@title Log in to Hugging Face
import os
from huggingface_hub import login, notebook_login, whoami

token = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass

if token:
    login(token=token, add_to_git_credential=False)
else:
    notebook_login()

print("Logged in as:", whoami().get("name"))


# 3. Load the MiniT2I Diffusers Pipeline

The Hugging Face repository stores both MiniT2I-B/16 and MiniT2I-L/16 weights. The `model_type` argument in the sampling cells selects which model to use.


In [ ]:
#@title Load pipeline
import torch
from diffusers import DiffusionPipeline

repo_id = "MiniT2I/MiniT2I"
pipe = DiffusionPipeline.from_pretrained(
    repo_id,
    custom_pipeline=repo_id,
    torch_dtype=dtype,
    trust_remote_code=True,
)
pipe = pipe.to(device)
print("Loaded", repo_id)


# 4. Generate an Image

Start with MiniT2I-B/16 for a quick sanity check. MiniT2I-L/16 generally benefits from a higher guidance scale and takes longer.


In [ ]:
#@title Sample one prompt
prompt = "A lonely astronaut standing on a quiet beach under two moons." #@param {type:"string"}
model_type = "b16" #@param ["b16", "l16"]
num_inference_steps = 40 #@param {type:"slider", min:1, max:100, step:1}
guidance_scale = 2.5 #@param {type:"number"}
seed = 0 #@param {type:"integer"}

generator = torch.Generator(device=device).manual_seed(seed) if device == "cuda" else torch.Generator().manual_seed(seed)
image = pipe(
    prompt,
    model_type=model_type,
    guidance_scale=guidance_scale,
    num_inference_steps=num_inference_steps,
    generator=generator,
).images[0]

output_path = f"minit2i_{model_type}_seed{seed}.png"
image.save(output_path)
print("Saved", output_path)
display(image)


# 5. Optional MiniT2I-B/16 vs. MiniT2I-L/16 Comparison

This cell generates one MiniT2I-B/16 image and one MiniT2I-L/16 image with their default demo guidance scales. It is slower than the single-image sanity check.


In [ ]:
#@title Generate MiniT2I-B/16 and MiniT2I-L/16 examples
run_comparison = False #@param {type:"boolean"}
comparison_prompt = "a transparent green backpack on a marble pedestal, with notebooks and a metal water bottle visible inside" #@param {type:"string"}
comparison_steps = 100 #@param {type:"slider", min:1, max:100, step:1}
comparison_seed = 123 #@param {type:"integer"}

if run_comparison:
    for mt, gs in [("b16", 2.5), ("l16", 6.0)]:
        gen = torch.Generator(device=device).manual_seed(comparison_seed) if device == "cuda" else torch.Generator().manual_seed(comparison_seed)
        img = pipe(
            comparison_prompt,
            model_type=mt,
            guidance_scale=gs,
            num_inference_steps=comparison_steps,
            generator=gen,
        ).images[0]
        path = f"minit2i_{mt}_comparison.png"
        img.save(path)
        print(f"{mt}: saved {path}")
        display(img)
else:
    print("Set run_comparison=True to generate both models.")
